# Week 3 BBO notebook – human-clean GP + UCB exploration

This notebook reads `Week03.csv` for the Week 1 and Week 2 query points and outputs, then combines that with the original `F*_initial_inputs.npy` and `F*_initial_outputs.npy` files.

It uses a **reproducible GP + UCB + novelty bonus** strategy, but searches over a **rounded candidate pool** so the final points are human-clean and easy to read.

This is **not** force-fitting any past answers. The clean points come from the candidate-generation rule itself:

1. sample random candidates in `[0, 1]^d` with a fixed seed
2. round each coordinate to the nearest grid step (default `0.01`)
3. discard candidates that are too close to previously tested points
4. score candidates using `mean + beta*std + novelty_bonus`

Scoring function:

$$\text{score}(x) = \mu(x) + \beta\sigma(x) + \lambda\,d(x, X_{seen})\,\mathrm{std}(y)$$

where `d(x, X_seen)` is the minimum normalised Euclidean distance from the candidate to any previously evaluated point.


In [1]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
data_dir = Path('data')
week_df = pd.read_csv(data_dir / 'weekly_results/Week03.csv')
week_df


,week,function,y,d,x1,x2,x3,x4,x5,x6,x7,x8
0,1,1,3.100722e-34,2,0.759779,0.804108,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,5.697925e-01,2,0.717915,0.002064,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,-1.518442e-01,3,0.994942,0.051967,0.792526,NaN,NaN,NaN,NaN,NaN
3,1,4,-7.158151e-01,4,0.404477,0.413254,0.303108,0.434359,NaN,NaN,NaN,NaN
4,1,5,3.653951e+03,4,0.940282,0.064909,0.998637,0.996528,NaN,NaN,NaN,NaN
5,1,6,-1.362062e+00,5,0.379776,0.684419,0.068171,0.984146,0.290503,NaN,NaN,NaN
6,1,7,2.395347e-01,6,0.015298,0.590564,0.658382,0.751493,0.425786,0.776978,NaN,NaN
7,1,8,9.589938e+00,8,0.034492,0.437681,0.011459,0.323393,0.989664,0.045780,0.097175,0.702465
8,2,1,-1.512885e-27,2,0.803235,0.719438,NaN,NaN,NaN,NaN,NaN,NaN
9,2,2,-7.246455e-02,2,0.957387,0.923033,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, week_df):
    X0, y0 = load_initial(function_id)
    rows = week_df[week_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    X_week = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    y_week = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, X_week])
    y = np.concatenate([y0, y_week])
    return X, y, d


In [4]:
def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)

def suggest_human_clean_point(function_id, week_df, beta, novelty_weight, n_candidates, seed, grid_step=0.01, min_sep=0.03):
    X, y, d = load_combined(function_id, week_df)

    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.05, 2.0))
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-2))
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed,
        n_restarts_optimizer=1,
    )

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)

    rng = np.random.RandomState(seed + function_id)
    candidates = rng.uniform(0.0, 1.0, size=(n_candidates, d))

    # Snap to a human-readable grid so outputs look clean by construction.
    candidates = np.round(candidates / grid_step) * grid_step
    candidates = np.clip(candidates, 0.0, 1.0)
    candidates = np.unique(candidates, axis=0)

    # Compute normalised distance to the already-evaluated set.
    dist = np.sqrt(((candidates[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist.min(axis=1)

    # Remove points that are too close to anything already tested.
    keep = min_dist >= min_sep
    candidates = candidates[keep]
    min_dist = min_dist[keep]

    mean, std = gp.predict(candidates, return_std=True)
    score = mean + beta * std + novelty_weight * np.std(y) * min_dist

    best_idx = np.argmax(score)
    x_best = candidates[best_idx]

    return {
        'function': function_id,
        'd': d,
        'x': x_best,
        'formatted': format_point(x_best),
        'mean_pred': float(mean[best_idx]),
        'std_pred': float(std[best_idx]),
        'min_dist': float(min_dist[best_idx]),
        'score': float(score[best_idx]),
        'kernel': str(gp.kernel_),
    }
